In [ ]:
%pip install scikit-learn

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

print("Scikit-learn version:", sklearn.__version__)


In [ ]:
current_directory = Path.cwd()

if current_directory.name in ("notebooks", "src"):
    project_root = current_directory.parent
else:
    project_root = current_directory

model_data_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_model.csv"
)

model_df = pd.read_csv(model_data_path)

print("Dataset path:", model_data_path)
print("Dataset shape:", model_df.shape)

display(model_df.head())

In [ ]:
target_column = "churn_value"

leakage_columns = [
    "churn_label",
    "churn_score",
    "churn_reason"
]

identifier_columns = [
    "customerid"
]

assert target_column in model_df.columns
assert model_df[target_column].isin([0, 1]).all()
assert model_df[target_column].isna().sum() == 0
assert not set(leakage_columns).intersection(model_df.columns)
assert not set(identifier_columns).intersection(model_df.columns)

print("Model dataset validation passed.")

In [ ]:
target_summary = (
    model_df[target_column]
    .value_counts()
    .sort_index()
    .rename_axis("churn_value")
    .reset_index(name="customer_count")
)

target_summary["percentage"] = (
    target_summary["customer_count"]
    / len(model_df)
    * 100
).round(2)

display(target_summary)

In [ ]:
X = model_df.drop(columns=[target_column]).copy()
y = model_df[target_column].astype(int).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

print("\nTraining target distribution:")
print(
    y_train.value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nTest target distribution:")
print(
    y_test.value_counts(normalize=True)
    .sort_index()
    .round(4)
)

In [ ]:
split_membership = pd.DataFrame(
    {
        "row_index": model_df.index,
        "dataset_split": np.where(
            model_df.index.isin(X_test.index),
            "test",
            "train"
        )
    }
)

print(split_membership["dataset_split"].value_counts())
display(split_membership.head())

In [ ]:
model_output_directory = (
    project_root
    / "reports"
    / "model_results"
)

model_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

split_membership.to_csv(
    model_output_directory / "train_test_split.csv",
    index=False
)

In [ ]:
numeric_features = (
    X_train.select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)

categorical_features = (
    X_train.select_dtypes(
        include=["object", "category", "bool"]
    )
    .columns
    .tolist()
)

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

In [ ]:
selected_features = (
    numeric_features
    + categorical_features
)

assert len(selected_features) == X_train.shape[1]
assert set(selected_features) == set(X_train.columns)

print("All model features have been classified.")

In [ ]:
def build_preprocessor():
    """
    Build preprocessing for numerical and categorical features.
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="if_binary"
                )
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop"
    )

    return preprocessor

In [ ]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [ ]:
dummy_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_preprocessor()
        ),
        (
            "classifier",
            DummyClassifier(
                strategy="prior"
            )
        )
    ]
)

In [ ]:
dummy_cv_results = cross_validate(
    estimator=dummy_pipeline,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

dummy_cv_results

In [ ]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_preprocessor()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

In [ ]:
logistic_cv_results = cross_validate(
    estimator=logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

logistic_cv_results

In [ ]:
def summarize_cv_results(
    model_name,
    cv_results,
    scoring_dictionary
):
    """
    Convert cross-validation results into one summary row.
    """

    summary = {
        "model": model_name
    }

    for metric_name in scoring_dictionary:
        fold_scores = cv_results[
            f"test_{metric_name}"
        ]

        summary[f"{metric_name}_mean"] = (
            fold_scores.mean()
        )

        summary[f"{metric_name}_std"] = (
            fold_scores.std()
        )

    summary["fit_time_mean"] = (
        cv_results["fit_time"].mean()
    )

    return summary

In [ ]:
cv_comparison = pd.DataFrame(
    [
        summarize_cv_results(
            "Dummy Classifier",
            dummy_cv_results,
            scoring
        ),
        summarize_cv_results(
            "Logistic Regression",
            logistic_cv_results,
            scoring
        )
    ]
)

numeric_columns = cv_comparison.select_dtypes(
    include="number"
).columns

cv_comparison[numeric_columns] = (
    cv_comparison[numeric_columns]
    .round(4)
)

display(cv_comparison)

In [ ]:
compact_cv_comparison = cv_comparison[
    [
        "model",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "roc_auc_mean",
        "pr_auc_mean"
    ]
].copy()

display(compact_cv_comparison)

In [ ]:
compact_cv_comparison.to_csv(
    model_output_directory
    / "baseline_logistic_cv_comparison.csv",
    index=False
)

print("Model comparison saved to:")
print(model_output_directory)

In [ ]:
metrics_to_plot = [
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
    "pr_auc_mean"
]

plot_data = (
    compact_cv_comparison
    .set_index("model")[metrics_to_plot]
    .T
)

plot_data.index = [
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC"
]

plot_data.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Cross-Validation Performance Comparison")
plt.xlabel("Metric")
plt.ylabel("Mean Cross-Validation Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Model")

plt.tight_layout()
plt.show()

In [ ]:
logistic_pipeline.fit(
    X_train,
    y_train
)

print("Logistic Regression fitted on the full training set.")

In [ ]:
fitted_preprocessor = (
    logistic_pipeline
    .named_steps["preprocessor"]
)

encoded_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

print("Features before encoding:", X_train.shape[1])
print("Features after encoding:", len(encoded_feature_names))

for feature_name in encoded_feature_names:
    print(feature_name)

In [ ]:
transformed_sample = (
    fitted_preprocessor
    .transform(X_train.iloc[:5])
)

if hasattr(transformed_sample, "toarray"):
    transformed_sample = (
        transformed_sample.toarray()
    )

transformed_sample_df = pd.DataFrame(
    transformed_sample,
    columns=encoded_feature_names,
    index=X_train.iloc[:5].index
)

display(transformed_sample_df)

In [ ]:
models_directory = (
    project_root
    / "models"
)

models_directory.mkdir(
    parents=True,
    exist_ok=True
)

logistic_model_path = (
    models_directory
    / "logistic_regression_candidate.joblib"
)

joblib.dump(
    logistic_pipeline,
    logistic_model_path
)

print("Candidate model saved to:")
print(logistic_model_path)

In [ ]:
%pip install --upgrade xgboost

In [ ]:
import json
import shutil
import subprocess

import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("XGBoost version:", xgb.__version__)

In [ ]:
xgboost_major_version = int(
    xgb.__version__.split(".")[0]
)

if xgboost_major_version < 2:
    raise RuntimeError(
        "Please upgrade XGBoost. The device='cuda' syntax "
        "requires XGBoost 2.0 or newer."
    )

In [ ]:
nvidia_smi_path = shutil.which("nvidia-smi")

if nvidia_smi_path is None:
    print(
        "nvidia-smi was not found. Check whether the NVIDIA "
        "driver is correctly installed."
    )
else:
    result = subprocess.run(
        [
            nvidia_smi_path,
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True,
        check=False
    )

    print("Detected NVIDIA GPU:")
    print(result.stdout)

    if result.stderr:
        print("nvidia-smi message:")
        print(result.stderr)

In [ ]:
def test_xgboost_gpu():
    """
    Train a tiny XGBoost model and inspect the device actually used.
    """

    X_probe = np.array(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
            [1.0, 1.0],
            [2.0, 0.0],
            [2.0, 1.0]
        ],
        dtype=np.float32
    )

    y_probe = np.array(
        [0, 0, 0, 1, 1, 1],
        dtype=np.int32
    )

    probe_model = XGBClassifier(
        n_estimators=5,
        max_depth=2,
        tree_method="hist",
        device="cuda",
        eval_metric="logloss",
        random_state=42,
        verbosity=0
    )

    try:
        probe_model.fit(X_probe, y_probe)

        configuration = json.loads(
            probe_model
            .get_booster()
            .save_config()
        )

        actual_device = (
            configuration["learner"]
            ["generic_param"]
            ["device"]
        )

        gpu_used = str(actual_device).startswith("cuda")

        return gpu_used, actual_device, None

    except Exception as error:
        return False, None, str(error)

In [ ]:
gpu_available, actual_device, gpu_error = (
    test_xgboost_gpu()
)

if gpu_available:
    print(
        "XGBoost GPU test passed.",
        "Device used:",
        actual_device
    )
else:
    print("XGBoost could not use the GPU.")

    if gpu_error is not None:
        print("\nError:")
        print(gpu_error)

In [ ]:
XGBOOST_DEVICE = (
    "cuda"
    if gpu_available
    else "cpu"
)

print("Selected XGBoost device:", XGBOOST_DEVICE)

In [ ]:
def build_tree_preprocessor():
    """
    Build preprocessing for tree-based models.

    Numeric features:
        median imputation only

    Categorical features:
        most-frequent imputation
        one-hot encoding
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="if_binary"
                )
            )
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop"
    )

In [ ]:
random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_tree_preprocessor()
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_split=4,
                min_samples_leaf=2,
                max_features="sqrt",
                class_weight=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest_pipeline

In [ ]:
random_forest_cv_results = cross_validate(
    estimator=random_forest_pipeline,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    scoring=scoring,
    return_train_score=False,

    # Run folds sequentially because each forest itself
    # already uses all CPU cores through n_jobs=-1.
    n_jobs=1
)

random_forest_cv_results

In [ ]:
negative_customers = int(
    (y_train == 0).sum()
)

positive_customers = int(
    (y_train == 1).sum()
)

class_ratio = (
    negative_customers
    / positive_customers
)

print("Non-churned training customers:", negative_customers)
print("Churned training customers:", positive_customers)
print("Negative-to-positive ratio:", round(class_ratio, 3))

In [ ]:
xgboost_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_tree_preprocessor()
        ),
        (
            "classifier",
            XGBClassifier(
                objective="binary:logistic",

                n_estimators=500,
                learning_rate=0.05,
                max_depth=4,
                min_child_weight=2,

                subsample=0.85,
                colsample_bytree=0.85,

                reg_alpha=0.1,
                reg_lambda=1.0,

                scale_pos_weight=1.0,

                tree_method="hist",
                device=XGBOOST_DEVICE,

                eval_metric="logloss",
                random_state=42,
                n_jobs=1,
                verbosity=0
            )
        )
    ]
)

xgboost_pipeline

In [ ]:
xgboost_cv_results = cross_validate(
    estimator=xgboost_pipeline,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    scoring=scoring,
    return_train_score=False,

    # Important: do not launch five folds simultaneously
    # on the same GPU.
    n_jobs=1
)

xgboost_cv_results

In [ ]:
all_cv_comparison = pd.DataFrame(
    [
        summarize_cv_results(
            "Dummy Classifier",
            dummy_cv_results,
            scoring
        ),
        summarize_cv_results(
            "Logistic Regression",
            logistic_cv_results,
            scoring
        ),
        summarize_cv_results(
            "Random Forest",
            random_forest_cv_results,
            scoring
        ),
        summarize_cv_results(
            "XGBoost GPU",
            xgboost_cv_results,
            scoring
        )
    ]
)

In [ ]:
numeric_result_columns = (
    all_cv_comparison
    .select_dtypes(include="number")
    .columns
)

all_cv_comparison[
    numeric_result_columns
] = (
    all_cv_comparison[
        numeric_result_columns
    ]
    .round(4)
)

In [ ]:
compact_model_comparison = (
    all_cv_comparison[
        [
            "model",
            "accuracy_mean",
            "precision_mean",
            "recall_mean",
            "f1_mean",
            "roc_auc_mean",
            "pr_auc_mean",
            "fit_time_mean"
        ]
    ]
    .sort_values(
        "pr_auc_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

display(compact_model_comparison)

In [ ]:
metrics_to_plot = [
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
    "pr_auc_mean"
]

comparison_plot_data = (
    all_cv_comparison
    .set_index("model")[metrics_to_plot]
    .T
)

comparison_plot_data.index = [
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC"
]

comparison_plot_data.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title(
    "Five-Fold Cross-Validation Model Comparison"
)

plt.xlabel("Evaluation Metric")
plt.ylabel("Mean Cross-Validation Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
training_time_comparison = (
    all_cv_comparison[
        [
            "model",
            "fit_time_mean"
        ]
    ]
    .sort_values(
        "fit_time_mean",
        ascending=True
    )
)

display(training_time_comparison)

In [ ]:
plt.figure(figsize=(9, 5))

bars = plt.barh(
    training_time_comparison["model"],
    training_time_comparison["fit_time_mean"]
)

plt.title("Average Model Fit Time per Cross-Validation Fold")
plt.xlabel("Average Fit Time in Seconds")
plt.ylabel("Model")

for bar, value in zip(
    bars,
    training_time_comparison["fit_time_mean"]
):
    plt.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f" {value:.3f}s",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
random_forest_pipeline.fit(
    X_train,
    y_train
)

print(
    "Random Forest fitted on the complete training set."
)

In [ ]:
xgboost_pipeline.fit(
    X_train,
    y_train
)

print(
    f"XGBoost fitted using device: "
    f"{XGBOOST_DEVICE}"
)

In [ ]:
fitted_xgboost_classifier = (
    xgboost_pipeline
    .named_steps["classifier"]
)

xgboost_configuration = json.loads(
    fitted_xgboost_classifier
    .get_booster()
    .save_config()
)

trained_device = (
    xgboost_configuration["learner"]
    ["generic_param"]
    ["device"]
)

print(
    "Device recorded in trained XGBoost model:",
    trained_device
)

In [ ]:
random_forest_model_path = (
    models_directory
    / "random_forest_candidate.joblib"
)

xgboost_model_path = (
    models_directory
    / "xgboost_gpu_candidate.joblib"
)

joblib.dump(
    random_forest_pipeline,
    random_forest_model_path
)

joblib.dump(
    xgboost_pipeline,
    xgboost_model_path
)

compact_model_comparison.to_csv(
    model_output_directory
    / "all_models_cv_comparison.csv",
    index=False
)

print("Random Forest saved to:")
print(random_forest_model_path)

print("\nXGBoost saved to:")
print(xgboost_model_path)

print("\nModel comparison saved to:")
print(
    model_output_directory
    / "all_models_cv_comparison.csv"
)

In [ ]:
import json

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import cross_val_predict

In [ ]:
candidate_models = {
    "Logistic Regression": logistic_pipeline,
    "Random Forest": random_forest_pipeline,
    "XGBoost GPU": xgboost_pipeline
}

In [ ]:
selection_table = (
    all_cv_comparison[
        all_cv_comparison["model"] != "Dummy Classifier"
    ]
    .sort_values(
        by=["pr_auc_mean", "roc_auc_mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    selection_table[
        [
            "model",
            "precision_mean",
            "recall_mean",
            "f1_mean",
            "roc_auc_mean",
            "pr_auc_mean"
        ]
    ]
)

In [ ]:
selected_model_name = selection_table.loc[0, "model"]

selected_pipeline = clone(
    candidate_models[selected_model_name]
)

print("Selected model:", selected_model_name)

In [ ]:
oof_probabilities = cross_val_predict(
    estimator=selected_pipeline,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    method="predict_proba",

    # Sequential folds prevent several models from
    # competing for the same GPU or CPU resources.
    n_jobs=1
)[:, 1]

print("Number of OOF probabilities:", len(oof_probabilities))
print("Minimum probability:", oof_probabilities.min())
print("Maximum probability:", oof_probabilities.max())

In [ ]:
assert len(oof_probabilities) == len(y_train)
assert np.all(
    (oof_probabilities >= 0)
    & (oof_probabilities <= 1)
)

print("OOF probability validation passed.")

In [ ]:
oof_roc_auc = roc_auc_score(
    y_train,
    oof_probabilities
)

oof_pr_auc = average_precision_score(
    y_train,
    oof_probabilities
)

print(f"OOF ROC-AUC: {oof_roc_auc:.4f}")
print(f"OOF PR-AUC:  {oof_pr_auc:.4f}")

In [ ]:
def evaluate_thresholds(
    y_true,
    probabilities,
    thresholds=None
):
    """
    Evaluate classification performance at different
    probability thresholds.
    """

    if thresholds is None:
        thresholds = np.arange(
            0.05,
            0.96,
            0.01
        )

    rows = []

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            predictions,
            labels=[0, 1]
        ).ravel()

        rows.append(
            {
                "threshold": threshold,
                "accuracy": accuracy_score(
                    y_true,
                    predictions
                ),
                "precision": precision_score(
                    y_true,
                    predictions,
                    zero_division=0
                ),
                "recall": recall_score(
                    y_true,
                    predictions,
                    zero_division=0
                ),
                "f1": f1_score(
                    y_true,
                    predictions,
                    zero_division=0
                ),
                "f2": fbeta_score(
                    y_true,
                    predictions,
                    beta=2,
                    zero_division=0
                ),
                "true_negatives": tn,
                "false_positives": fp,
                "false_negatives": fn,
                "true_positives": tp
            }
        )

    return pd.DataFrame(rows)

In [ ]:
threshold_results = evaluate_thresholds(
    y_true=y_train,
    probabilities=oof_probabilities
)

display(
    threshold_results.head()
)

In [ ]:
best_f1_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_f2_row = threshold_results.loc[
    threshold_results["f2"].idxmax()
]

threshold_comparison = pd.DataFrame(
    [
        {
            "method": "Default threshold",
            "threshold": 0.50
        },
        {
            "method": "Maximum F1",
            "threshold": best_f1_row["threshold"]
        },
        {
            "method": "Maximum F2",
            "threshold": best_f2_row["threshold"]
        }
    ]
)

display(threshold_comparison)

In [ ]:
display(
    threshold_results[
        (threshold_results["threshold"] >= 0.20)
        & (threshold_results["threshold"] <= 0.70)
    ][
        [
            "threshold",
            "precision",
            "recall",
            "f1",
            "f2",
            "false_positives",
            "false_negatives"
        ]
    ].round(4)
)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    threshold_results["threshold"],
    threshold_results["precision"],
    label="Precision"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["recall"],
    label="Recall"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["f1"],
    label="F1"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["f2"],
    label="F2"
)

plt.axvline(
    x=0.50,
    linestyle="--",
    label="Default threshold"
)

plt.xlabel("Probability Threshold")
plt.ylabel("Metric Score")
plt.title("Precision–Recall Trade-off by Classification Threshold")
plt.ylim(0, 1)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
threshold_selection_metric = "f2"

In [ ]:
selected_threshold_row = threshold_results.loc[
    threshold_results[
        threshold_selection_metric
    ].idxmax()
]

selected_threshold = float(
    selected_threshold_row["threshold"]
)

print(
    "Threshold selection metric:",
    threshold_selection_metric.upper()
)

print(
    "Selected threshold:",
    round(selected_threshold, 3)
)

display(
    selected_threshold_row.to_frame(
        name="value"
    )
)

In [ ]:
def classification_metrics_at_threshold(
    y_true,
    probabilities,
    threshold,
    model_label
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": model_label,
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            predictions,
            beta=2,
            zero_division=0
        ),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    }

In [ ]:
oof_threshold_comparison = pd.DataFrame(
    [
        classification_metrics_at_threshold(
            y_true=y_train,
            probabilities=oof_probabilities,
            threshold=0.50,
            model_label="Default threshold"
        ),
        classification_metrics_at_threshold(
            y_true=y_train,
            probabilities=oof_probabilities,
            threshold=selected_threshold,
            model_label="Tuned threshold"
        )
    ]
)

display(
    oof_threshold_comparison.round(4)
)

In [ ]:
false_negative_cost = 5
false_positive_cost = 1

In [ ]:
threshold_results["estimated_cost"] = (
    threshold_results["false_negatives"]
    * false_negative_cost
    +
    threshold_results["false_positives"]
    * false_positive_cost
)

lowest_cost_row = threshold_results.loc[
    threshold_results["estimated_cost"].idxmin()
]

print(
    "Lowest-cost threshold:",
    round(lowest_cost_row["threshold"], 3)
)

display(
    lowest_cost_row.to_frame(
        name="value"
    )
)

In [ ]:
final_model = clone(
    candidate_models[selected_model_name]
)

final_model.fit(
    X_train,
    y_train
)

print(
    f"{selected_model_name} fitted on the complete "
    f"training dataset."
)

In [ ]:
test_probabilities = final_model.predict_proba(
    X_test
)[:, 1]

default_test_predictions = (
    test_probabilities >= 0.50
).astype(int)

tuned_test_predictions = (
    test_probabilities >= selected_threshold
).astype(int)

In [ ]:
test_evaluation = pd.DataFrame(
    [
        classification_metrics_at_threshold(
            y_true=y_test,
            probabilities=test_probabilities,
            threshold=0.50,
            model_label=(
                f"{selected_model_name} — Default"
            )
        ),
        classification_metrics_at_threshold(
            y_true=y_test,
            probabilities=test_probabilities,
            threshold=selected_threshold,
            model_label=(
                f"{selected_model_name} — Tuned"
            )
        )
    ]
)

test_evaluation["roc_auc"] = roc_auc_score(
    y_test,
    test_probabilities
)

test_evaluation["pr_auc"] = (
    average_precision_score(
        y_test,
        test_probabilities
    )
)

display(
    test_evaluation.round(4)
)

In [ ]:
print(
    f"Selected threshold: "
    f"{selected_threshold:.3f}\n"
)

print(
    classification_report(
        y_test,
        tuned_test_predictions,
        target_names=[
            "Not Churned",
            "Churned"
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
tuned_confusion_matrix = confusion_matrix(
    y_test,
    tuned_test_predictions,
    labels=[0, 1]
)

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=tuned_confusion_matrix,
    display_labels=[
        "Not Churned",
        "Churned"
    ]
)

display_matrix.plot(
    values_format="d"
)

plt.title(
    f"Test Confusion Matrix\n"
    f"Threshold = {selected_threshold:.2f}"
)

plt.tight_layout()
plt.show()

In [ ]:
false_positive_rate, true_positive_rate, _ = (
    roc_curve(
        y_test,
        test_probabilities
    )
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

plt.figure(figsize=(8, 6))

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"ROC-AUC = {test_roc_auc:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC Curve")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_precision, test_recall, _ = (
    precision_recall_curve(
        y_test,
        test_probabilities
    )
)

test_pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

baseline_precision = y_test.mean()

plt.figure(figsize=(8, 6))

plt.plot(
    test_recall,
    test_precision,
    label=f"PR-AUC = {test_pr_auc:.3f}"
)

plt.axhline(
    y=baseline_precision,
    linestyle="--",
    label=(
        f"Churn prevalence = "
        f"{baseline_precision:.3f}"
    )
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Test Precision–Recall Curve")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
enriched_analytics_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_analytics_enriched.csv"
)

base_analytics_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_analytics.csv"
)

if enriched_analytics_path.exists():
    analytics_lookup = pd.read_csv(
        enriched_analytics_path
    )
else:
    analytics_lookup = pd.read_csv(
        base_analytics_path
    )

assert len(analytics_lookup) == len(model_df)

print("Analytics lookup loaded.")

In [ ]:
test_predictions_df = analytics_lookup.loc[
    X_test.index,
    [
        "customerid",
        "cltv",
        "monthly_charges",
        "contract",
        "tenure_months"
    ]
].copy()

test_predictions_df["actual_churn"] = (
    y_test.loc[X_test.index].values
)

test_predictions_df["churn_probability"] = (
    test_probabilities
)

test_predictions_df["predicted_churn_default"] = (
    default_test_predictions
)

test_predictions_df["predicted_churn_tuned"] = (
    tuned_test_predictions
)

test_predictions_df["selected_threshold"] = (
    selected_threshold
)

In [ ]:
test_predictions_df["risk_segment"] = pd.cut(
    test_predictions_df["churn_probability"],
    bins=[
        -np.inf,
        0.30,
        0.60,
        np.inf
    ],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

In [ ]:
test_predictions_df["priority_score"] = (
    test_predictions_df["churn_probability"]
    * test_predictions_df["cltv"]
)

In [ ]:
display(
    test_predictions_df
    .sort_values(
        "priority_score",
        ascending=False
    )
    .head(20)
)

In [ ]:
final_model_path = (
    models_directory
    / "final_churn_model.joblib"
)

joblib.dump(
    final_model,
    final_model_path
)

In [ ]:
model_metadata = {
    "model_name": selected_model_name,
    "target": target_column,
    "selected_threshold": selected_threshold,
    "threshold_selection_metric": (
        threshold_selection_metric
    ),
    "test_roc_auc": float(test_roc_auc),
    "test_pr_auc": float(test_pr_auc),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "random_state": 42
}

metadata_path = (
    models_directory
    / "final_churn_model_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_metadata,
        file,
        indent=4
    )

In [ ]:
threshold_results.to_csv(
    model_output_directory
    / "threshold_analysis.csv",
    index=False
)

oof_threshold_comparison.to_csv(
    model_output_directory
    / "oof_threshold_comparison.csv",
    index=False
)

test_evaluation.to_csv(
    model_output_directory
    / "final_test_evaluation.csv",
    index=False
)

test_predictions_df.to_csv(
    model_output_directory
    / "test_customer_predictions.csv",
    index=False
)

In [ ]:
print("Final model saved to:")
print(final_model_path)

print("\nModel metadata saved to:")
print(metadata_path)

print("\nTest predictions saved to:")
print(
    model_output_directory
    / "test_customer_predictions.csv"
)